# Chapter 20 Companion Notebook: Text Pre-processing and Data Quality in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch20_Text_Preprocessing_and_Data_Quality.ipynb)

This notebook accompanies Chapter 20 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses a synthetic business text corpus so students can run the workflow in Colab without customer data, paid APIs, or external files. The examples are intentionally small, transparent, and business-oriented.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Treat markdown sections as short lecture notes and code sections as live demos. The dataset is synthetic, but the workflow mirrors what analysts do with reviews, support tickets, chat logs, open-ended survey responses, and public brand mentions. If a cell runs slowly, keep `FAST_MODE = True` and reduce `N_RECORDS` in the setup cell.


## Why this matters (business framing)

Text pre-processing is the measurement layer of text mining. Before a model can discover topics, classify sentiment, retrieve similar complaints, or summarize customer needs, the analyst has already made many measurement choices: what counts as one observation, which metadata travel with the text, what gets preserved, what gets standardized, what gets replaced, and what gets removed.

In marketing analytics, these choices are not cosmetic. Removing negation can flip sentiment. Keeping long order IDs can make a model memorize irrelevant uniqueness. Leaving templates inside support tickets can make topic models cluster documents by agent macros rather than customer needs. Fitting a vocabulary on held-out text can leak evaluation information. This notebook turns those issues into a practical workflow that students can audit and explain.


## Agenda

1. Setup and reproducibility
2. Synthetic business text corpus with metadata
3. Unit of analysis and corpus context
4. Data quality diagnostics: missingness, length, duplicates, templates, drift, and leakage
5. Cleaning and normalization policy
6. Pattern-based extraction and privacy-aware analysis layers
7. Vocabulary control for sparse baselines
8. Leakage-safe TF-IDF evaluation
9. Tokenization, segmentation, and chunking
10. Governance artifacts and exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to define a unit of analysis for business text, run a corpus sanity check, detect exact and near duplicates, identify template and leakage risks, write a defensible cleaning policy, preserve negation and intensity cues, replace sensitive patterns with placeholders, fit TF-IDF features without leaking held-out information, compare vocabulary control choices, design chunks that map back to source documents, and create a minimal preprocessing specification for governance.


## Connection map

Chapters 21 through 25 build on the analysis-ready corpus created here. Embeddings, topic modeling, sentiment classification, transformer-based NLP, and retrieval-augmented text mining all depend on the same early decisions: which text span enters the model, which artifacts are removed, which identifiers are protected, and whether the pipeline respects the decision moment. Chapter 20 is therefore not a preliminary chore. It is the measurement foundation for the entire text mining part of the book.


In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================
import os
import re
import sys
import json
import math
import time
import random
import hashlib
import warnings
import importlib
import subprocess
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)

SEED = 20
FAST_MODE = True
N_RECORDS = 850 if FAST_MODE else 1800
OUTPUT_DIR = Path("ch20_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


set_seed(SEED)
print(f"Ready. FAST_MODE={FAST_MODE}, N_RECORDS={N_RECORDS}, output folder='{OUTPUT_DIR}'")


## Utility functions

The helpers below keep the main sections focused on measurement decisions. They handle text display, normalization, duplicate detection, simple plotting, and evaluation summaries.


In [ ]:
# ============================================================
# Utility functions for display, plotting, text processing, and diagnostics
# ============================================================

def print_section(title):
    print("=" * len(title))
    print(title)
    print("=" * len(title))


def stable_hash(text, n=12):
    text = "" if pd.isna(text) else str(text)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def normalize_unicode(text):
    if pd.isna(text):
        return ""
    # Normalize visually similar characters. Also fix a few common mojibake artifacts.
    text = str(text)
    replacements = {
        "â€™": "'",
        "â€œ": '"',
        "â€�": '"',
        "â€“": "-",
        "Ã©": "e",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return unicodedata.normalize("NFKC", text)


def word_tokens(text):
    text = normalize_unicode(text).lower()
    return re.findall(r"[a-z][a-z']+|<[^>]+>|\d+", text)


def char_shingles(text, k=5):
    text = re.sub(r"\s+", " ", normalize_unicode(text).lower()).strip()
    if len(text) < k:
        return {text} if text else set()
    return {text[i:i+k] for i in range(len(text) - k + 1)}


def jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def show_examples(df, cols, n=5, random_state=SEED):
    display(df[cols].sample(min(n, len(df)), random_state=random_state).reset_index(drop=True))


def plot_bar(series, title, xlabel="", ylabel="Count", top_n=None):
    s = series.copy()
    if top_n is not None:
        s = s.head(top_n)
    plt.figure(figsize=(8, 4.5))
    s.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=35, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


def evaluate_binary_classifier(name, y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    out = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "positive_rate_pred": float(y_pred.mean()),
        "positive_rate_actual": float(np.mean(y_true)),
    }
    return out


def top_terms_from_linear_model(vectorizer, model, n=12):
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = model.coef_.ravel()
    top_pos = pd.DataFrame({"term": feature_names[np.argsort(coefs)[-n:][::-1]], "coefficient": np.sort(coefs)[-n:][::-1]})
    top_neg = pd.DataFrame({"term": feature_names[np.argsort(coefs)[:n]], "coefficient": np.sort(coefs)[:n]})
    return top_pos, top_neg


## 2. Synthetic business text corpus with metadata

The dataset below mimics a common marketing analytics corpus. Each row is a text observation connected to metadata such as channel, date, customer segment, product, language, case ID, and future outcome. The corpus intentionally includes problems that analysts must diagnose before modeling: missing text, exact duplicates, near duplicates, boilerplate templates, mixed languages, encoding artifacts, long identifiers, personal information, and post-outcome notes.


In [ ]:
# ============================================================
# Generate a synthetic business text corpus
# ============================================================
set_seed(SEED)
rng = np.random.default_rng(SEED)

channels = ["review", "survey", "support_ticket", "chat", "social_post"]
segments = ["new", "loyal", "at_risk", "enterprise"]
products = ["AeroFit", "NovaBlend", "HomeBase Hub", "FreshBox", "PixelLamp"]
regions = ["West", "South", "Midwest", "Northeast"]
issues = [
    "delivery_delay", "login_problem", "billing_confusion", "product_quality",
    "feature_request", "competitor_mention", "praise", "cancellation_risk"
]
competitors = ["RivalCo", "QuickCart", "BrightHome", "FitPlus"]

issue_base_prob = np.array([0.16, 0.14, 0.12, 0.13, 0.11, 0.08, 0.18, 0.08])
issue_base_prob = issue_base_prob / issue_base_prob.sum()

def random_date():
    start = np.datetime64("2024-01-01")
    end = np.datetime64("2025-07-01")
    days = int((end - start).astype(int))
    return pd.Timestamp(start + np.timedelta64(int(rng.integers(0, days)), "D"))


def order_id():
    return f"#{rng.choice(['A','B','Z','Q'])}{rng.integers(1000,9999)}-{rng.integers(10,99)}"


def phone_number():
    return f"555-{rng.integers(100,999)}-{rng.integers(1000,9999)}"


def promo_code():
    return f"SAVE{rng.integers(10,99)}" if rng.random() < 0.6 else f"WELCOME{rng.integers(100,999)}"


def issue_sentence(issue, product, competitor):
    oid = order_id()
    if issue == "delivery_delay":
        return f"My order {oid} was late and the box arrived damaged. I still need help."
    if issue == "login_problem":
        return "Can't log in!!! VERY frustrated because the app says invalid code."
    if issue == "billing_confusion":
        return f"The invoice looks wrong and the price increase was not explained. Promo {promo_code()} did not apply."
    if issue == "product_quality":
        return f"The {product} stopped working after two weeks. This is not acceptable."
    if issue == "feature_request":
        return f"Please add dark mode, family sharing, and clearer notifications for {product}."
    if issue == "competitor_mention":
        return f"I might switch to {competitor} because their support feels faster."
    if issue == "praise":
        return f"Not bad at all. Delivery was quick, setup was easy, and {product} works great :)"
    if issue == "cancellation_risk":
        return "I am tired of waiting and may cancel if this is not fixed today."
    return "General feedback."


def channel_style(channel, sentence, product, issue):
    if channel == "review":
        stars = 5 if issue == "praise" else int(rng.choice([1, 2, 3, 4], p=[0.28, 0.32, 0.25, 0.15]))
        return f"I bought {product}. {sentence} Rating: {stars}/5."
    if channel == "survey":
        return f"When asked why, the customer wrote: {sentence}"
    if channel == "support_ticket":
        return f"Thank you for contacting support. {sentence} Case note: customer asked for an update."
    if channel == "chat":
        return f"Customer: {sentence}\nAgent: I understand and will check the account.\nCustomer: Please hurry."
    if channel == "social_post":
        return f"@CompanyHelp {sentence} #customerexperience"
    return sentence


def maybe_add_noise(text, issue, product, dt, future_churn):
    # Encoding, HTML, IDs, URLs, PII, and templates.
    if rng.random() < 0.07:
        text = text.replace("Can't", "Canâ€™t").replace("quick", "quick!!!")
    if rng.random() < 0.10:
        text += f" Email me at customer{rng.integers(100,999)}@example.com."
    if rng.random() < 0.08:
        text += f" Call {phone_number()} after 5pm."
    if rng.random() < 0.12:
        text += f" Details are at https://example.com/help/{rng.integers(10000,99999)}."
    if rng.random() < 0.09:
        text = f"<p>{text}</p>"
    if rng.random() < 0.15:
        text += " This message may contain confidential information and is intended only for the recipient."
    if rng.random() < 0.10:
        text += " Thanks, Support Team. Your satisfaction is important to us."
    if rng.random() < 0.04:
        text = text[: rng.integers(35, max(45, min(len(text), 100)))] + "..."
    # New vocabulary after a product launch creates drift.
    if dt >= pd.Timestamp("2025-01-01") and product == "AeroFit" and rng.random() < 0.55:
        text += " The new AeroFit Pro strap feels different."
    # Post-outcome leakage phrases. These would not be available at the decision moment.
    leak = False
    if future_churn and rng.random() < 0.42:
        text += " Resolution note: customer cancelled subscription after refund issued."
        leak = True
    elif issue in ["delivery_delay", "login_problem", "billing_confusion"] and rng.random() < 0.10:
        text += " Internal note: escalated to Tier 2."
        leak = True
    return text, leak

rows = []
for i in range(N_RECORDS):
    dt = random_date()
    channel = rng.choice(channels, p=[0.27, 0.15, 0.25, 0.20, 0.13])
    segment = rng.choice(segments, p=[0.32, 0.34, 0.22, 0.12])
    product = rng.choice(products)
    region = rng.choice(regions)
    language = rng.choice(["en", "es", "mixed"], p=[0.90, 0.06, 0.04])
    issue_probs = issue_base_prob.copy()
    if segment == "at_risk":
        issue_probs[issues.index("cancellation_risk")] += 0.10
        issue_probs[issues.index("praise")] -= 0.08
    if channel in ["support_ticket", "chat"]:
        issue_probs[issues.index("login_problem")] += 0.05
        issue_probs[issues.index("delivery_delay")] += 0.04
        issue_probs[issues.index("praise")] -= 0.06
    issue_probs = np.clip(issue_probs, 0.01, None)
    issue_probs = issue_probs / issue_probs.sum()
    issue = rng.choice(issues, p=issue_probs)
    competitor = rng.choice(competitors)

    churn_logit = -2.2
    churn_logit += 1.05 if segment == "at_risk" else 0.0
    churn_logit += 1.00 if issue in ["cancellation_risk", "billing_confusion", "product_quality"] else 0.0
    churn_logit += 0.50 if channel in ["support_ticket", "chat"] else 0.0
    churn_logit -= 1.25 if issue == "praise" else 0.0
    future_churn = rng.random() < (1 / (1 + np.exp(-churn_logit)))

    sentence = issue_sentence(issue, product, competitor)
    if language == "es":
        sentence = "No puedo resolver esto. " + sentence
    elif language == "mixed":
        sentence = "Necesito ayuda. " + sentence
    text = channel_style(channel, sentence, product, issue)
    text, leak = maybe_add_noise(text, issue, product, dt, future_churn)
    if rng.random() < 0.025:
        text = "" if rng.random() < 0.55 else None

    rows.append({
        "doc_id": f"D{i:05d}",
        "customer_id": f"C{rng.integers(1000, 1210)}",
        "case_id": f"CASE{rng.integers(20000, 20580)}" if channel in ["support_ticket", "chat"] else None,
        "date": dt,
        "month": str(dt.to_period("M")),
        "channel": channel,
        "segment": segment,
        "product": product,
        "region": region,
        "language": language,
        "primary_issue": issue,
        "text_raw": text,
        "future_churn": int(future_churn),
        "post_outcome_leak_inserted": int(leak),
    })

corpus = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

# Add exact duplicates and near-duplicates to make the diagnostics meaningful.
exact_dups = corpus.sample(25, random_state=SEED).copy()
exact_dups["doc_id"] = [f"DX{i:04d}" for i in range(len(exact_dups))]
exact_dups["customer_id"] = [f"C{rng.integers(1210, 1300)}" for _ in range(len(exact_dups))]

near_dups = corpus.sample(25, random_state=SEED + 1).copy()
near_dups["doc_id"] = [f"DN{i:04d}" for i in range(len(near_dups))]
near_dups["text_raw"] = near_dups["text_raw"].fillna("").astype(str) + " Thank you."

corpus = pd.concat([corpus, exact_dups, near_dups], ignore_index=True).sort_values("date").reset_index(drop=True)
corpus["text_hash_raw"] = corpus["text_raw"].apply(stable_hash)
corpus["n_chars_raw"] = corpus["text_raw"].fillna("").astype(str).str.len()
corpus["is_missing_text"] = corpus["text_raw"].isna() | (corpus["text_raw"].astype(str).str.strip() == "")

print_section("Synthetic corpus created")
print(corpus.shape)
display(corpus.head(8)[["doc_id", "date", "channel", "segment", "product", "language", "primary_issue", "future_churn", "text_raw"]])


In [ ]:
# A quick profile of the corpus by source channel.
channel_profile = (
    corpus.groupby("channel")
    .agg(
        records=("doc_id", "count"),
        customers=("customer_id", "nunique"),
        missing_text=("is_missing_text", "sum"),
        avg_chars=("n_chars_raw", "mean"),
        churn_rate=("future_churn", "mean"),
    )
    .sort_values("records", ascending=False)
)
channel_profile["avg_chars"] = channel_profile["avg_chars"].round(1)
channel_profile["churn_rate"] = channel_profile["churn_rate"].round(3)
display(channel_profile)

plot_bar(channel_profile["records"], "Corpus coverage by text source", xlabel="Channel", ylabel="Records")


## 3. Unit of analysis and corpus context

A business text corpus is hierarchical. A row may be a review, a survey answer, a support ticket, a chat turn, or a social post, but the business decision may happen at the customer, case, product, or month level. The safest workflow preserves identifiers so the analyst can process at one level and report at another.


In [ ]:
# ============================================================
# Unit-of-analysis checks
# ============================================================
unit_summary = pd.DataFrame({
    "level": ["document or message", "customer", "case", "product", "customer-month", "product-month"],
    "identifier": ["doc_id", "customer_id", "case_id", "product", "customer_id + month", "product + month"],
    "unique_units": [
        corpus["doc_id"].nunique(),
        corpus["customer_id"].nunique(),
        corpus["case_id"].nunique(),
        corpus["product"].nunique(),
        corpus[["customer_id", "month"]].drop_duplicates().shape[0],
        corpus[["product", "month"]].drop_duplicates().shape[0],
    ],
    "why_it_matters": [
        "Main processing unit in this notebook",
        "Useful for churn and relationship decisions",
        "Useful for service escalation and resolution decisions",
        "Useful for product improvement and positioning decisions",
        "Useful for trend reporting without double-counting daily fragments",
        "Useful for comparing issue trends across product lines",
    ],
})
display(unit_summary)

records_per_customer = corpus.groupby("customer_id")["doc_id"].count()
print(f"Average records per customer: {records_per_customer.mean():.2f}")
print(f"Maximum records for one customer: {records_per_customer.max()}")
print("This reminds us not to treat every row as fully independent customer evidence.")


In [ ]:
# ============================================================
# Corpus sanity check: dates, channels, language, and missingness
# ============================================================
san_check = {
    "start_date": str(corpus["date"].min().date()),
    "end_date": str(corpus["date"].max().date()),
    "n_documents": int(corpus.shape[0]),
    "n_customers": int(corpus["customer_id"].nunique()),
    "n_products": int(corpus["product"].nunique()),
    "missing_text_rate": float(corpus["is_missing_text"].mean()),
    "non_english_or_mixed_rate": float((corpus["language"] != "en").mean()),
}
print(json.dumps(san_check, indent=2))

coverage = pd.crosstab(corpus["month"], corpus["channel"])
display(coverage.tail(8))

plt.figure(figsize=(9, 4.5))
coverage.plot(ax=plt.gca(), marker="o")
plt.title("Monthly text volume by channel")
plt.xlabel("Month")
plt.ylabel("Records")
plt.xticks(rotation=45, ha="right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 4. Text data quality diagnostics

The next cells operationalize the main risks discussed in Chapter 20: missing or clipped text, duplicate content, templated boilerplate, drift, and leakage. The purpose is not to reject the corpus. The purpose is to know what evidence the corpus can and cannot support.


In [ ]:
# ============================================================
# Length, missingness, and possible truncation diagnostics
# ============================================================
corpus["possibly_truncated"] = corpus["text_raw"].fillna("").astype(str).str.endswith("...")
length_diag = corpus.groupby("channel").agg(
    records=("doc_id", "count"),
    missing_text_rate=("is_missing_text", "mean"),
    possible_truncation_rate=("possibly_truncated", "mean"),
    median_chars=("n_chars_raw", "median"),
    p10_chars=("n_chars_raw", lambda s: np.percentile(s, 10)),
    p90_chars=("n_chars_raw", lambda s: np.percentile(s, 90)),
).round(3)
display(length_diag)

plt.figure(figsize=(8, 4.5))
corpus.loc[~corpus["is_missing_text"], "n_chars_raw"].hist(bins=35)
plt.title("Text length distribution")
plt.xlabel("Number of characters")
plt.ylabel("Documents")
plt.grid(alpha=0.25)
plt.show()

show_examples(corpus[corpus["possibly_truncated"]], ["doc_id", "channel", "n_chars_raw", "text_raw"], n=5)


In [ ]:
# ============================================================
# Exact duplicate detection
# ============================================================
nonmissing = corpus[~corpus["is_missing_text"]].copy()
exact_dup_groups = (
    nonmissing.groupby("text_hash_raw")
    .agg(records=("doc_id", "count"), example_text=("text_raw", "first"), channels=("channel", lambda x: ", ".join(sorted(set(x)))))
    .query("records > 1")
    .sort_values("records", ascending=False)
)
print(f"Exact duplicate groups: {len(exact_dup_groups)}")
print(f"Documents inside exact duplicate groups: {int(exact_dup_groups['records'].sum()) if len(exact_dup_groups) else 0}")
display(exact_dup_groups.head(8))


In [ ]:
# ============================================================
# Near-duplicate detection with character shingles and Jaccard overlap
# This classroom version samples documents so the comparison is easy to run.
# ============================================================
near_sample = nonmissing.sample(min(260, len(nonmissing)), random_state=SEED).reset_index(drop=True)
shingle_sets = [char_shingles(t, k=5) for t in near_sample["text_raw"]]
near_pairs = []
for i in range(len(near_sample)):
    for j in range(i + 1, len(near_sample)):
        score = jaccard(shingle_sets[i], shingle_sets[j])
        if score >= 0.82 and near_sample.loc[i, "text_hash_raw"] != near_sample.loc[j, "text_hash_raw"]:
            near_pairs.append({
                "doc_id_a": near_sample.loc[i, "doc_id"],
                "doc_id_b": near_sample.loc[j, "doc_id"],
                "jaccard_5gram": round(score, 3),
                "text_a": near_sample.loc[i, "text_raw"],
                "text_b": near_sample.loc[j, "text_raw"],
            })
near_pairs_df = pd.DataFrame(near_pairs).sort_values("jaccard_5gram", ascending=False)
print(f"Near-duplicate pairs in sample: {len(near_pairs_df)}")
display(near_pairs_df.head(6))


In [ ]:
# ============================================================
# Template and boilerplate diagnostics
# ============================================================
template_patterns = {
    "confidentiality_disclaimer": r"confidential information|intended only for the recipient",
    "support_greeting": r"thank you for contacting support",
    "support_signature": r"thanks, support team|your satisfaction is important",
    "agent_turn": r"agent: i understand",
}

for name, pattern in template_patterns.items():
    corpus[f"template_{name}"] = corpus["text_raw"].fillna("").str.lower().str.contains(pattern, regex=True).astype(int)

template_cols = [c for c in corpus.columns if c.startswith("template_")]
template_summary = corpus.groupby("channel")[template_cols].mean().round(3)
display(template_summary)

plot_bar(corpus[template_cols].mean().sort_values(ascending=False), "Share of records containing common templates", xlabel="Template pattern", ylabel="Share")

show_examples(corpus[corpus[template_cols].sum(axis=1) > 0], ["doc_id", "channel", "text_raw"], n=4)


In [ ]:
# ============================================================
# Drift diagnostics: vocabulary and issue mix over time
# ============================================================
corpus["period"] = np.where(corpus["date"] < pd.Timestamp("2025-01-01"), "before_2025", "after_2025")
issue_mix = pd.crosstab(corpus["month"], corpus["primary_issue"], normalize="index")

plt.figure(figsize=(9, 4.5))
for issue in ["delivery_delay", "login_problem", "product_quality", "praise", "cancellation_risk"]:
    plt.plot(issue_mix.index.astype(str), issue_mix[issue], marker="o", label=issue)
plt.title("Monthly issue mix")
plt.xlabel("Month")
plt.ylabel("Share of monthly corpus")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

before_tokens = Counter()
after_tokens = Counter()
for _, row in corpus[~corpus["is_missing_text"]].iterrows():
    target = before_tokens if row["period"] == "before_2025" else after_tokens
    target.update(word_tokens(row["text_raw"]))

vocab_drift = []
for tok in set(before_tokens) | set(after_tokens):
    before_rate = before_tokens[tok] / max(1, sum(before_tokens.values()))
    after_rate = after_tokens[tok] / max(1, sum(after_tokens.values()))
    if before_tokens[tok] + after_tokens[tok] >= 8:
        vocab_drift.append({"token": tok, "before_count": before_tokens[tok], "after_count": after_tokens[tok], "after_minus_before_rate": after_rate - before_rate})

vocab_drift_df = pd.DataFrame(vocab_drift).sort_values("after_minus_before_rate", ascending=False)
print("Tokens that became more common after 2025:")
display(vocab_drift_df.head(12))


In [ ]:
# ============================================================
# Leakage diagnostics: phrases that appear after the business decision moment
# ============================================================
leakage_patterns = {
    "cancelled_subscription": r"cancelled subscription|customer cancelled",
    "refund_issued": r"refund issued",
    "escalated_tier2": r"escalated to tier 2",
    "resolution_note": r"resolution note|internal note",
}
for name, pattern in leakage_patterns.items():
    corpus[f"leak_{name}"] = corpus["text_raw"].fillna("").str.lower().str.contains(pattern, regex=True).astype(int)

leak_cols = [c for c in corpus.columns if c.startswith("leak_")]
leak_summary = corpus.groupby("future_churn")[leak_cols].mean().round(3)
display(leak_summary)

print("Records with at least one leakage phrase:", int((corpus[leak_cols].sum(axis=1) > 0).sum()))
show_examples(corpus[corpus[leak_cols].sum(axis=1) > 0], ["doc_id", "date", "future_churn", "text_raw"], n=5)


## 5. Core cleaning and normalization policy

Cleaning fixes integrity problems. Normalization reduces unhelpful variation while preserving variation that carries meaning. The policy below keeps raw text for audit, standardizes Unicode and whitespace, removes HTML artifacts, replaces sensitive patterns with placeholders, preserves negation, and creates separate features for intensity cues.


In [ ]:
# ============================================================
# Mini example: aggressive cleaning versus context-aware cleaning
# ============================================================
mini_comments = pd.DataFrame({
    "raw": [
        "Can't log in!!! VERY frustrated",
        "Not bad, delivery was quick",
        "Order #Z9812 still not resolved.",
    ]
})

aggressive_stopwords = set("a an and are as at be by for from has have i in is it of on or that the this to was were with not still".split())

def aggressive_clean(text):
    text = normalize_unicode(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    toks = [t for t in text.split() if t not in aggressive_stopwords and len(t) > 1]
    return " ".join(toks)


def caps_ratio(text):
    letters = re.findall(r"[A-Za-z]", normalize_unicode(text))
    if not letters:
        return 0.0
    caps = [ch for ch in letters if ch.isupper()]
    return len(caps) / len(letters)


def context_aware_clean(text):
    text = normalize_unicode(text)
    cap = caps_ratio(text)
    text = re.sub(r"#[A-Z0-9-]{4,}", " <ID> ", text)
    text = re.sub(r"!{2,}", " ! ", text)
    text = re.sub(r"[^A-Za-z0-9<>'!\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return f"{text} [caps_ratio={cap:.2f}]"

mini_comments["aggressive_clean"] = mini_comments["raw"].apply(aggressive_clean)
mini_comments["context_aware_clean"] = mini_comments["raw"].apply(context_aware_clean)
display(mini_comments)


In [ ]:
# ============================================================
# A reusable cleaning policy for this chapter
# ============================================================
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.I)
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", flags=re.I)
PHONE_RE = re.compile(r"\b(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b")
ORDER_RE = re.compile(r"#?[A-Z]{1,3}\d{3,5}(?:[-_]\d{2,4})?\b")
PROMO_RE = re.compile(r"\b(?:SAVE|WELCOME|PROMO)\d{2,4}\b", flags=re.I)
HTML_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(r"[\U0001F300-\U0001FAFF]")
REPEAT_PUNCT_RE = re.compile(r"([!?])\1+")
OUTCOME_SENTENCE_RE = re.compile(
    r"(?:resolution note|internal note)\s*:\s*[^.]*?(?:cancelled subscription|refund issued|escalated to tier 2)[^.]*\.?,?",
    flags=re.I,
)

NEGATIONS = {"no", "not", "never", "cannot", "can't", "dont", "don't", "didnt", "didn't", "without"}


def extract_text_features(text):
    text = "" if pd.isna(text) else str(text)
    normalized = normalize_unicode(text)
    return {
        "raw_caps_ratio": caps_ratio(normalized),
        "raw_repeat_punct_count": len(REPEAT_PUNCT_RE.findall(normalized)),
        "raw_has_emoji": int(bool(EMOJI_RE.search(normalized))),
        "raw_url_count": len(URL_RE.findall(normalized)),
        "raw_email_count": len(EMAIL_RE.findall(normalized)),
        "raw_phone_count": len(PHONE_RE.findall(normalized)),
        "raw_order_id_count": len(ORDER_RE.findall(normalized)),
        "raw_promo_count": len(PROMO_RE.findall(normalized)),
    }


def clean_text(text, remove_outcome_leak=False, lowercase=True):
    text = "" if pd.isna(text) else str(text)
    text = normalize_unicode(text)
    if remove_outcome_leak:
        text = OUTCOME_SENTENCE_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" <URL> ", text)
    text = EMAIL_RE.sub(" <EMAIL> ", text)
    text = PHONE_RE.sub(" <PHONE> ", text)
    text = PROMO_RE.sub(" <PROMO> ", text)
    text = ORDER_RE.sub(" <ID> ", text)
    text = REPEAT_PUNCT_RE.sub(r" \1 ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if lowercase:
        text = text.lower()
    return text

feature_df = pd.DataFrame([extract_text_features(t) for t in corpus["text_raw"]])
corpus_clean = pd.concat([corpus.copy(), feature_df], axis=1)
corpus_clean["text_clean_with_leak"] = corpus_clean["text_raw"].apply(lambda x: clean_text(x, remove_outcome_leak=False))
corpus_clean["text_clean_safe"] = corpus_clean["text_raw"].apply(lambda x: clean_text(x, remove_outcome_leak=True))
corpus_clean["clean_hash_safe"] = corpus_clean["text_clean_safe"].apply(stable_hash)
corpus_clean["n_tokens_safe"] = corpus_clean["text_clean_safe"].apply(lambda x: len(word_tokens(x)))

cols = ["doc_id", "text_raw", "text_clean_with_leak", "text_clean_safe", "raw_caps_ratio", "raw_email_count", "raw_order_id_count"]
show_examples(corpus_clean[~corpus_clean["is_missing_text"]], cols, n=6)


In [ ]:
# Cleaning changes measurable corpus properties.
cleaning_summary = pd.DataFrame({
    "measure": [
        "nonempty raw documents",
        "nonempty cleaned documents",
        "records with URLs",
        "records with email addresses",
        "records with phone numbers",
        "records with order-like IDs",
        "records with repeated punctuation",
        "records with emojis",
        "records with detected leakage phrases",
    ],
    "value": [
        int((~corpus_clean["is_missing_text"]).sum()),
        int((corpus_clean["text_clean_safe"].str.len() > 0).sum()),
        int((corpus_clean["raw_url_count"] > 0).sum()),
        int((corpus_clean["raw_email_count"] > 0).sum()),
        int((corpus_clean["raw_phone_count"] > 0).sum()),
        int((corpus_clean["raw_order_id_count"] > 0).sum()),
        int((corpus_clean["raw_repeat_punct_count"] > 0).sum()),
        int((corpus_clean["raw_has_emoji"] > 0).sum()),
        int((corpus_clean[leak_cols].sum(axis=1) > 0).sum()),
    ]
})
display(cleaning_summary)


## 6. Pattern-based extraction and privacy-aware analysis layers

Pattern-based extraction gives text a small amount of reliable structure. It is especially useful for privacy, operational joins, campaign analysis, and dashboards. The key governance idea is to separate a restricted raw-text layer from an analysis-ready layer.


In [ ]:
# ============================================================
# Build an analysis-ready layer with extracted fields and placeholders
# ============================================================
analysis_ready = corpus_clean[[
    "doc_id", "customer_id", "case_id", "date", "month", "channel", "segment", "product", "region", "language", "primary_issue", "future_churn",
    "text_hash_raw", "clean_hash_safe", "text_clean_safe", "n_tokens_safe",
    "raw_caps_ratio", "raw_repeat_punct_count", "raw_has_emoji", "raw_url_count", "raw_email_count", "raw_phone_count", "raw_order_id_count", "raw_promo_count"
]].copy()

analysis_ready["has_pii_pattern"] = ((analysis_ready["raw_email_count"] + analysis_ready["raw_phone_count"]) > 0).astype(int)
analysis_ready["has_operational_id"] = (analysis_ready["raw_order_id_count"] > 0).astype(int)
analysis_ready["has_link"] = (analysis_ready["raw_url_count"] > 0).astype(int)
analysis_ready["has_promo"] = (analysis_ready["raw_promo_count"] > 0).astype(int)

privacy_summary = analysis_ready[["has_pii_pattern", "has_operational_id", "has_link", "has_promo"]].mean().sort_values(ascending=False).to_frame("share_of_records")
display(privacy_summary.round(3))

show_examples(analysis_ready, ["doc_id", "channel", "has_pii_pattern", "has_operational_id", "has_link", "has_promo", "text_clean_safe"], n=6)


In [ ]:
# ============================================================
# Dictionary-based business concepts and competitor mentions
# ============================================================
concept_dictionary = {
    "delivery_issue": ["late", "delivery", "box arrived damaged", "arrived damaged"],
    "login_issue": ["log in", "invalid code", "can't log"],
    "billing_issue": ["invoice", "price increase", "promo", "did not apply"],
    "quality_issue": ["stopped working", "not acceptable", "broken"],
    "feature_request": ["please add", "dark mode", "family sharing", "notifications"],
    "cancellation_language": ["cancel", "tired of waiting", "not fixed today"],
    "praise_language": ["not bad", "works great", "setup was easy", "quick"],
}
competitor_dictionary = ["rivalco", "quickcart", "brighthome", "fitplus"]


def contains_any(text, phrases):
    text = str(text).lower()
    return int(any(p.lower() in text for p in phrases))

for concept, phrases in concept_dictionary.items():
    analysis_ready[f"concept_{concept}"] = analysis_ready["text_clean_safe"].apply(lambda x, p=phrases: contains_any(x, p))
analysis_ready["mentions_competitor"] = analysis_ready["text_clean_safe"].apply(lambda x: contains_any(x, competitor_dictionary))

concept_cols = [c for c in analysis_ready.columns if c.startswith("concept_")] + ["mentions_competitor"]
concept_summary = analysis_ready.groupby("channel")[concept_cols].mean().round(3)
display(concept_summary)

plot_bar(analysis_ready[concept_cols].mean().sort_values(ascending=False), "Dictionary concept prevalence", xlabel="Concept", ylabel="Share of records")


## 7. Vocabulary control for sparse baselines

Sparse methods such as bag-of-words and TF-IDF are still useful because they are fast, transparent, and strong baselines. The vocabulary is the measurement instrument. The next cells show how rare-token filtering, common-token filtering, n-grams, and vocabulary caps change what the model can see.


In [ ]:
# ============================================================
# Compare vocabulary control choices
# ============================================================
nonempty = analysis_ready[analysis_ready["text_clean_safe"].str.len() > 0].copy()
texts = nonempty["text_clean_safe"].tolist()

vectorizer_specs = [
    ("unigrams_no_filters", CountVectorizer(ngram_range=(1, 1), token_pattern=r"(?u)\b\w[\w']+\b")),
    ("unigrams_min_df_3", CountVectorizer(ngram_range=(1, 1), min_df=3, token_pattern=r"(?u)\b\w[\w']+\b")),
    ("unigrams_bigrams_min_df_3", CountVectorizer(ngram_range=(1, 2), min_df=3, token_pattern=r"(?u)\b\w[\w']+\b")),
    ("tfidf_bigrams_capped", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.85, max_features=900, token_pattern=r"(?u)\b\w[\w']+\b")),
]

vocab_rows = []
for name, vec in vectorizer_specs:
    X = vec.fit_transform(texts)
    vocab = vec.get_feature_names_out()
    vocab_rows.append({
        "spec": name,
        "n_documents": X.shape[0],
        "n_features": X.shape[1],
        "matrix_density": X.nnz / (X.shape[0] * X.shape[1]),
        "example_features": ", ".join(vocab[:10]),
    })

vocab_summary = pd.DataFrame(vocab_rows)
display(vocab_summary)


In [ ]:
# Inspect the most frequent n-grams under a defensible sparse baseline.
cv = CountVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.90, max_features=900, token_pattern=r"(?u)\b\w[\w']+\b")
X_counts = cv.fit_transform(texts)
term_counts = np.asarray(X_counts.sum(axis=0)).ravel()
terms = np.array(cv.get_feature_names_out())
term_df = pd.DataFrame({"term": terms, "count": term_counts}).sort_values("count", ascending=False)

print("Most frequent retained terms and phrases:")
display(term_df.head(20))

print("Examples of low-frequency retained terms near the minimum threshold:")
display(term_df.sort_values("count").head(12))


## 8. Leakage-safe TF-IDF evaluation

This section demonstrates two leakage issues. First, post-outcome phrases inside the text can act like an answer key. Second, TF-IDF should be fit on training text only and then applied to held-out text. The performance comparison below uses the same future outcome target with and without post-outcome phrases.


In [ ]:
# ============================================================
# Time-respecting split and leakage-safe model evaluation
# ============================================================
model_df = corpus_clean[(corpus_clean["text_clean_safe"].str.len() > 0)].copy()
model_df = model_df.sort_values("date").reset_index(drop=True)
split_date = pd.Timestamp("2025-03-01")
train_df = model_df[model_df["date"] < split_date].copy()
test_df = model_df[model_df["date"] >= split_date].copy()

print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}, Split date: {split_date.date()}")
print(f"Train churn rate: {train_df['future_churn'].mean():.3f}, Test churn rate: {test_df['future_churn'].mean():.3f}")


def fit_tfidf_logit(train_text, train_y, test_text, test_y, name):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.85,
        max_features=1200,
        token_pattern=r"(?u)\b\w[\w']+\b",
    )
    X_train = vectorizer.fit_transform(train_text)
    X_test = vectorizer.transform(test_text)
    clf = LogisticRegression(max_iter=700, class_weight="balanced", random_state=SEED)
    clf.fit(X_train, train_y)
    prob = clf.predict_proba(X_test)[:, 1]
    metrics = evaluate_binary_classifier(name, test_y, prob)
    return metrics, vectorizer, clf, prob

metrics_leak, vec_leak, clf_leak, prob_leak = fit_tfidf_logit(
    train_df["text_clean_with_leak"], train_df["future_churn"],
    test_df["text_clean_with_leak"], test_df["future_churn"],
    "Text includes post-outcome leakage"
)
metrics_safe, vec_safe, clf_safe, prob_safe = fit_tfidf_logit(
    train_df["text_clean_safe"], train_df["future_churn"],
    test_df["text_clean_safe"], test_df["future_churn"],
    "Leakage phrases removed"
)

model_results = pd.DataFrame([metrics_leak, metrics_safe]).round(3)
display(model_results)

plt.figure(figsize=(7, 4.5))
plt.bar(model_results["model"], model_results["auc"])
plt.title("Leakage can inflate held-out performance")
plt.ylabel("AUC")
plt.xticks(rotation=20, ha="right")
plt.ylim(0.45, 1.0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Top terms also function as a quality diagnostic.
print_section("Top positive terms when leakage remains")
pos_leak, neg_leak = top_terms_from_linear_model(vec_leak, clf_leak, n=12)
display(pos_leak)

print_section("Top positive terms after leakage removal")
pos_safe, neg_safe = top_terms_from_linear_model(vec_safe, clf_safe, n=12)
display(pos_safe)

print("If answer-key phrases dominate the first list, the model is not learning early customer signals.")


In [ ]:
# ============================================================
# TF-IDF feature fitting: train-only versus all-corpus fitting
# ============================================================
right_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.85, max_features=1200, token_pattern=r"(?u)\b\w[\w']+\b")
right_vec.fit(train_df["text_clean_safe"])

wrong_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.85, max_features=1200, token_pattern=r"(?u)\b\w[\w']+\b")
wrong_vec.fit(model_df["text_clean_safe"])

right_vocab = set(right_vec.get_feature_names_out())
wrong_vocab = set(wrong_vec.get_feature_names_out())
wrong_only_terms = sorted(wrong_vocab - right_vocab)[:25]
shared_terms = sorted(list(right_vocab & wrong_vocab))[:12]

fit_audit = pd.DataFrame({
    "fit_policy": ["RIGHT: train text only", "WRONG: train + test text"],
    "n_features": [len(right_vocab), len(wrong_vocab)],
    "example_terms_available_only_when_test_is_used": ["", ", ".join(wrong_only_terms[:12])],
})
display(fit_audit)

idf_compare = []
right_idf = dict(zip(right_vec.get_feature_names_out(), right_vec.idf_))
wrong_idf = dict(zip(wrong_vec.get_feature_names_out(), wrong_vec.idf_))
for term in shared_terms:
    idf_compare.append({"term": term, "idf_train_only": right_idf[term], "idf_all_corpus": wrong_idf[term], "difference": wrong_idf[term] - right_idf[term]})
display(pd.DataFrame(idf_compare).round(3))

print("The right pattern is fit on train, transform train, transform held-out. The held-out text should not shape the measurement instrument.")


## 9. Tokenization, segmentation, and chunking

Modern NLP often uses fixed tokenizers and large context windows, but analysts still choose the span of text to process. Chunking should preserve coherent meaning while maintaining a mapping back to the decision unit. The same document can be split by sentences, paragraphs, or fixed-size windows with overlap.


In [ ]:
# ============================================================
# Tokenization and chunking utilities
# ============================================================
long_ticket = """
Header: Case CASE20441, Product AeroFit, Customer Segment: loyal.

Customer: I bought AeroFit last month. The first shipment was late, and the replacement strap does not fit. I still cannot log in to track the return.
Agent: Thank you for contacting support. I understand and will check the account.
Customer: Please do not send another generic response. I need a working strap before Friday because this is for a team event.

This message may contain confidential information and is intended only for the recipient.
Thanks, Support Team. Your satisfaction is important to us.
""".strip()


def simple_sentence_split(text):
    pieces = re.split(r"(?<=[.!?])\s+", normalize_unicode(text).strip())
    return [p.strip() for p in pieces if p.strip()]


def paragraph_chunks(text):
    pieces = re.split(r"\n\s*\n", normalize_unicode(text).strip())
    return [p.strip().replace("\n", " ") for p in pieces if p.strip()]


def fixed_size_chunks(text, max_words=35, overlap=8):
    toks = normalize_unicode(text).split()
    chunks = []
    start = 0
    step = max_words - overlap
    while start < len(toks):
        chunk = " ".join(toks[start:start + max_words])
        chunks.append(chunk)
        if start + max_words >= len(toks):
            break
        start += step
    return chunks

chunk_rows = []
for strategy, chunks in [
    ("sentence", simple_sentence_split(long_ticket)),
    ("paragraph", paragraph_chunks(long_ticket)),
    ("fixed_overlap", fixed_size_chunks(long_ticket, max_words=35, overlap=8)),
]:
    for i, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            "doc_id": "LONG_TICKET_001",
            "strategy": strategy,
            "chunk_id": f"{strategy}_{i}",
            "n_words": len(chunk.split()),
            "chunk_text": chunk,
        })

chunk_df = pd.DataFrame(chunk_rows)
display(chunk_df)

plt.figure(figsize=(8, 4.5))
chunk_df.groupby("strategy")["n_words"].apply(list).explode().astype(int).reset_index().boxplot(column="n_words", by="strategy", grid=False)
plt.suptitle("")
plt.title("Chunk size varies by strategy")
plt.xlabel("Strategy")
plt.ylabel("Words per chunk")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Chunk-level counting can overstate prevalence unless we aggregate back to documents.
# ============================================================
chunk_df["mentions_support_template"] = chunk_df["chunk_text"].str.lower().str.contains("thank you for contacting support|satisfaction is important").astype(int)
chunk_df["mentions_customer_problem"] = chunk_df["chunk_text"].str.lower().str.contains("late|strap|cannot log in|generic response").astype(int)

counting_demo = chunk_df.groupby("strategy").agg(
    chunks=("chunk_id", "count"),
    problem_mentions_by_chunk=("mentions_customer_problem", "sum"),
    template_mentions_by_chunk=("mentions_support_template", "sum"),
)
counting_demo["document_has_problem"] = 1
counting_demo["document_has_template"] = 1
counting_demo["warning"] = "Chunk counts are not document counts"
display(counting_demo)

print("Reporting should usually aggregate back to the decision unit, such as the document, case, customer, or product-month.")


## 10. Governance artifacts and reproducibility

A preprocessing pipeline should be versioned like any other measurement instrument. The artifact below records the corpus snapshot, unit of analysis, cleaning rules, placeholder policies, leakage controls, feature-fitting policy, and chunking strategy.


In [ ]:
# ============================================================
# Minimal preprocessing specification and quality card
# ============================================================
preprocess_spec = {
    "chapter": "20 Text Pre-processing and Data Quality",
    "corpus_version": "synthetic_ch20_corpus_v1",
    "preprocess_version": "preprocess_v1",
    "unit_of_analysis": "document_or_message",
    "decision_unit_examples": ["customer", "case", "product_month"],
    "raw_text_policy": "Keep restricted raw text for audit; use analysis-ready layer for modeling.",
    "unicode_policy": "NFKC normalization plus correction of common mojibake artifacts.",
    "placeholder_policy": {
        "urls": "<URL>",
        "emails": "<EMAIL>",
        "phones": "<PHONE>",
        "order_like_ids": "<ID>",
        "promo_codes": "<PROMO>",
    },
    "signal_preservation": ["negation", "capitalization ratio", "repeated punctuation count", "emoji indicator"],
    "template_policy": "Detect and report common boilerplate patterns before topic or similarity analysis.",
    "leakage_policy": "Remove post-outcome notes before predicting future churn; fit vectorizers on training text only.",
    "split_policy": f"Time-respecting split at {split_date.date()} for evaluation.",
    "chunking_policy": "Preserve natural structure when possible; keep chunk identifiers that map back to source documents.",
}
spec_string = json.dumps(preprocess_spec, sort_keys=True, indent=2)
preprocess_spec["spec_hash"] = stable_hash(spec_string, n=16)
print(json.dumps(preprocess_spec, indent=2))

quality_card = pd.DataFrame({
    "item": [
        "Documents", "Customers", "Date range", "Missing text rate", "Exact duplicate groups",
        "Records with possible truncation", "Records with PII patterns", "Records with leakage phrases", "Train/test split date", "Preprocess spec hash",
    ],
    "value": [
        len(corpus_clean), corpus_clean["customer_id"].nunique(), f"{corpus_clean['date'].min().date()} to {corpus_clean['date'].max().date()}",
        f"{corpus_clean['is_missing_text'].mean():.3f}", len(exact_dup_groups), int(corpus_clean["possibly_truncated"].sum()),
        int(analysis_ready["has_pii_pattern"].sum()), int((corpus_clean[leak_cols].sum(axis=1) > 0).sum()),
        str(split_date.date()), preprocess_spec["spec_hash"],
    ],
})
display(quality_card)


In [ ]:
# Save the shareable analysis layer and the governance files.
analysis_ready.to_csv(OUTPUT_DIR / "analysis_ready_text_sample.csv", index=False)
quality_card.to_csv(OUTPUT_DIR / "quality_card.csv", index=False)
with open(OUTPUT_DIR / "preprocess_spec.json", "w") as f:
    json.dump(preprocess_spec, f, indent=2)

print("Saved files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)


## Decision guide: what to check before modeling business text

A beginner-friendly text pipeline should answer five questions before modeling. First, what does one row represent, and how does it map back to the business decision unit? Second, what coverage limits, missingness, truncation, duplicates, templates, drift, and leakage risks appear in the corpus? Third, which elements should be preserved, standardized, or replaced? Fourth, which identifiers and sensitive strings need placeholders or restricted access? Fifth, was every feature mapping fit only on the data allowed at the decision moment?

When the answer to any question is unclear, pause before interpreting themes or model accuracy. In text analytics, measurement errors often look like interesting insights.


## Exercises

**Exercise 1. Unit of analysis.** Choose one business decision unit for this corpus, such as customer, case, product-month, or channel-month. Write a short explanation of why that unit matches a plausible marketing decision.

**Exercise 2. Cleaning policy.** Modify `clean_text()` so it preserves hashtags as a separate placeholder `<HASHTAG>`. Re-run the vocabulary summary and explain which features change.

**Exercise 3. Near duplicates.** Lower the near-duplicate threshold from 0.82 to 0.75. Inspect five additional pairs. Which pairs are true near duplicates, and which are only formulaic language?

**Exercise 4. Template handling.** Create a new `text_customer_voice_only` field that removes the confidentiality disclaimer and support signature. Compare the most frequent terms before and after removal.

**Exercise 5. Leakage audit.** Add one new leakage phrase pattern to `leakage_patterns`. Re-run the leakage summary and explain whether it is an answer key or a legitimate early signal.

**Exercise 6. Vocabulary control.** Fit a TF-IDF model with unigrams only and another with unigrams plus bigrams. Compare the top positive terms for churn. Which representation is more interpretable for a marketing manager?

**Exercise 7. Chunking.** Apply the chunking functions to three long support tickets from the synthetic corpus. Decide which chunking strategy you would use for retrieval and justify your choice.

**Exercise 8. Governance.** Edit `preprocess_spec` to reflect your revised choices. Generate a new `spec_hash` and explain why changing preprocessing should be treated as a new measurement version.
